# Spherical trên Swin đóng băng — 4 dataset (Val, 3 seed)

Thay **ViT đóng băng** bằng **Swin đóng băng** (Swin-B và Swin-T) trong nhánh Spherical; mọi thứ khác giữ nguyên (split của nhóm, spherical k-means, κ = κ_mult / median khoảng cách, cùng lưới k, κ_mult, w, λ, w_in).

| biến thể | nhánh Spherical | KAC |
|---|---|---|
| `proto` | backbone đóng băng, đứng một mình | không dùng |
| `fuse` | backbone đóng băng | logit KAC của APT-ViT (dump có sẵn) |
| `hier` | backbone đóng băng chọn task | KAC chọn lớp trong task |

Bảng in ra đặt ViT, Swin-B, Swin-T cạnh nhau trên cùng seed, cùng ảnh, cùng logit KAC, kèm chi phí (tham số, GFLOPs, ảnh/giây). Không train gì: mỗi ảnh chỉ đi qua mỗi backbone một lần.

**Kaggle:** Settings → GPU + Internet On. Add Input:
1. dataset CIFAR-100, CUB-200, Stanford Cars, ImageNet-R của nhóm;
2. output các notebook APT đã chạy (chứa `runs/kac/<dataset>/val/dumps/seed*`) cho đủ seed 0, 1, 2 của từng dataset. Một notebook có nhiều version: version cũ → tab Output → New dataset, rồi Add Input dataset đó.

Seed nào không có dump KAC thì chỉ tính được `proto` (notebook tự nhận ra, in "KHÔNG CÓ" trong log).


## 1. Cấu hình

In [1]:
# ==============================================================================
# 1. CẤU HÌNH
# ==============================================================================
DATASETS = ["CIFAR100", "CUB200", "StanfordCars", "ImageNet_R"]   # cả 4 trong một phiên (không train)
SEEDS    = [0, 1, 2]
PHASE    = "val"            # "val": tâm lớp từ 80% Train, đánh giá trên 20% Val (KHÔNG đụng Test)
# backbone đóng băng cho nhánh Spherical: {tên timm: nhãn trong bảng}
BACKBONES = {"swin_base_patch4_window7_224.ms_in22k_ft_in1k": "Swin-B",    # ~88M tham số, cùng cỡ ViT-B/16
             "swin_tiny_patch4_window7_224.ms_in1k":          "Swin-T"}    # ~28M tham số, nhẹ hơn ~3 lần
BATCH, WORKERS = 128, 4
OUT = "/kaggle/working/swin_spherical"
INPUT_ROOT = "/kaggle/input"
PRETRAINED = True           # chỉ để False khi chạy thử không có Internet
import os; os.makedirs(OUT, exist_ok=True)
assert PHASE in ("val", "test") and set(SEEDS) <= {0, 1, 2, 3, 4}


## 2. Dữ liệu và chia seed theo quy trình của nhóm

In [2]:
# ==============================================================================
# 2. DỮ LIỆU + CHIA SEED (y hệt check-dataset.ipynb / notebook APT của nhóm)
# ==============================================================================
import os, glob, json, pickle
import numpy as np
from torchvision.datasets import ImageFolder

DS_SPEC = {
    "CIFAR100":     dict(C=100, sizes=[10] * 10, kind="cifar",
                         roots=[f"{INPUT_ROOT}/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
                                f"{INPUT_ROOT}/cifar-100-python/cifar-100-python"]),
    "CUB200":       dict(C=200, sizes=[20] * 10, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/cyizhuo/cub-200-2011-by-classes-folder",
                                f"{INPUT_ROOT}/cub-200-2011-by-classes-folder"]),
    "StanfordCars": dict(C=196, sizes=[16] + [20] * 9, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
                                f"{INPUT_ROOT}/stanford-car-dataset-by-classes-folder/car_data/car_data"]),
    "ImageNet_R":   dict(C=200, sizes=[20] * 10, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/thohintrnhngc/imagenet-r", f"{INPUT_ROOT}/imagenet-r"]),
}

def find_root(ds):
    sp = DS_SPEC[ds]
    ok = (lambda p: all(os.path.isfile(os.path.join(p, f)) for f in ("train", "test", "meta"))) if sp["kind"] == "cifar" \
         else (lambda p: os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test")))
    for p in sp["roots"]:
        if ok(p): return p
    tail = os.path.basename(sp["roots"][0].rstrip("/"))
    for p in glob.glob(f"{INPUT_ROOT}/**/{tail}", recursive=True):
        if ok(p): return p
    raise FileNotFoundError(f"{ds}: không thấy dữ liệu. Add Input dataset của nhóm ({sp['roots'][0]})")

def setup_seed(train_targets, seed, num_classes, task_sizes, n_splits=5, base_seed=1993):
    class_order = np.random.RandomState(seed).permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur: cur + sz]); cur += sz
    targets = np.asarray(train_targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        perm = rng_split.permutation(np.where(targets == c)[0])
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed: tr_idx.extend(folds[k])
    return task_classes, np.array(tr_idx), np.array(va_idx)

SRC = {}
def load_source(ds):
    """S["tr"], S["te"]: đường dẫn ảnh (folder) hoặc mảng ảnh uint8 (CIFAR); S["tr_id"]: tên ảnh dùng để đối chiếu json."""
    if ds in SRC: return SRC[ds]
    root, sp = find_root(ds), DS_SPEC[ds]
    if sp["kind"] == "cifar":
        S = dict(root=root)
        for s, name in (("tr", "train"), ("te", "test")):
            with open(os.path.join(root, name), "rb") as f: e = pickle.load(f, encoding="latin1")
            S[s] = e["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1); S[s + "_y"] = [int(v) for v in e["fine_labels"]]
            S[s + "_id"] = [f"{name}:{i}" for i in range(len(S[s + "_y"]))]
    else:
        tr, te = ImageFolder(os.path.join(root, "train")), ImageFolder(os.path.join(root, "test"))
        assert tr.classes == te.classes, f"{ds}: tên lớp train/test không khớp"
        S = dict(root=root, tr=[p for p, _ in tr.samples], tr_y=list(tr.targets), te=[p for p, _ in te.samples], te_y=list(te.targets))
        for s in ("tr", "te"): S[s + "_id"] = [os.path.relpath(p, root) for p in S[s]]
    assert len(set(S["tr_y"])) == sp["C"], f"{ds}: số lớp {len(set(S['tr_y']))} ≠ {sp['C']}"
    SRC[ds] = S
    print(f"{ds}: {root} | train {len(S['tr_y'])} ảnh, test {len(S['te_y'])} ảnh")
    return S

def seed_split(ds, seed):
    """(train_items, train_labels, eval_items, eval_labels) theo đúng thứ tự file json của nhóm.
    item = (split, chỉ số ảnh), split ∈ {"tr","te"}; nhãn = vị trí lớp trong thứ tự task."""
    sp, S = DS_SPEC[ds], load_source(ds)
    task_classes, tr_idx, va_idx = setup_seed(S["tr_y"], seed, sp["C"], sp["sizes"])
    pos = {c: i for i, c in enumerate(sum(task_classes, []))}
    if PHASE == "val":
        tr = [("tr", int(i)) for i in tr_idx]; ev = [("tr", int(i)) for i in va_idx]
    else:
        tr = [("tr", i) for i in range(len(S["tr_y"]))]; ev = [("te", i) for i in range(len(S["te_y"]))]
    lab = lambda it: pos[S[it[0] + "_y"][it[1]]]
    return tr, [lab(x) for x in tr], ev, [lab(x) for x in ev]

def check_against_json(ds, seed, ev, ev_y):
    """Nếu output phiên APT (thư mục splits) được Add Input: so danh sách ảnh đánh giá với file json đã dùng khi train KAC."""
    hits = glob.glob(f"{INPUT_ROOT}/**/splits/{ds}/{PHASE}/seed{seed}/test.json", recursive=True)
    if not hits: return "không có json để đối chiếu"
    js = json.load(open(hits[0])); S = load_source(ds)
    rel = (lambda p: p) if DS_SPEC[ds]["kind"] == "cifar" else \
          (lambda p: os.path.relpath(p, js["root"]) if p.startswith(js["root"]) else p)
    mine = [S[s + "_id"][i] for s, i in ev]
    assert [rel(p) for p in js["data"]] == mine and js["targets"] == ev_y, f"{ds} seed {seed}: split khác file json của phiên APT"
    return "khớp file json của phiên APT"


## 3. Trích đặc trưng Swin đóng băng

In [3]:
# ==============================================================================
# 3. TRÍCH ĐẶC TRƯNG BACKBONE ĐÓNG BĂNG (mỗi ảnh một lần, dùng chung cho mọi seed) + đo chi phí
# ==============================================================================
import time, torch, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

class Imgs(Dataset):
    def __init__(self, src, tf): self.src, self.tf = src, tf
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        x = self.src[i]
        if isinstance(x, str):
            with Image.open(x) as im: return self.tf(im.convert("RGB"))
        return self.tf(Image.fromarray(x))                                   # CIFAR: mảng 32x32x3

MODELS = {}
def get_model(name):
    if name not in MODELS:
        for k in list(MODELS): del MODELS[k]                                  # giữ 1 model trên GPU
        torch.cuda.empty_cache() if DEV == "cuda" else None
        m = timm.create_model(name, pretrained=PRETRAINED, num_classes=0).eval().to(DEV)   # num_classes=0 -> vector sau pooling
        cfg = timm.data.resolve_data_config({}, model=m)
        MODELS[name] = (m, timm.data.create_transform(**cfg))
        print(f"{name}: {m.num_features} chiều | tiền xử lý {cfg['input_size']}, crop_pct {cfg.get('crop_pct')}")
    return MODELS[name]

@torch.no_grad()
def extract(name, src):
    m, tf = get_model(name); out, t0 = [], time.time()
    dl = DataLoader(Imgs(src, tf), batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=(DEV == "cuda"))
    for b, x in enumerate(dl):
        with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == "cuda")):
            out.append(m(x.to(DEV, non_blocking=True)).float().cpu())
        if b % 50 == 0: print(f"   {min((b + 1) * BATCH, len(src))}/{len(src)} ảnh, {time.time() - t0:.0f}s", flush=True)
    return torch.cat(out).half()

FEAT = {}
def features(ds, name):
    """{"tr": tensor, "te": tensor}; cache ở OUT/feats để chạy lại không phải trích lần nữa."""
    if (ds, name) in FEAT: return FEAT[(ds, name)]
    S = load_source(ds); FEAT[(ds, name)] = {}
    for s in (["tr"] if PHASE == "val" else ["tr", "te"]):
        p = f"{OUT}/feats/{ds}_{s}_{name}.pt"
        if os.path.exists(p) and torch.load(p)["ids"] == S[s + "_id"]:
            FEAT[(ds, name)][s] = torch.load(p)["f"]; print(f"{ds}/{s}/{BACKBONES.get(name, name)}: dùng cache"); continue
        print(f"{ds}/{s}/{BACKBONES.get(name, name)}: trích {len(S[s + '_y'])} ảnh ...", flush=True)
        FEAT[(ds, name)][s] = extract(name, S[s]); os.makedirs(os.path.dirname(p), exist_ok=True)
        torch.save({"ids": S[s + "_id"], "f": FEAT[(ds, name)][s]}, p)
    return FEAT[(ds, name)]

@torch.no_grad()
def cost(name, label):
    """Số tham số, GFLOPs / ảnh 224x224, tốc độ forward thuần (fp16, batch BATCH, không tính đọc ảnh)."""
    m = timm.create_model(name, pretrained=False, num_classes=0).eval().to(DEV)
    x1 = torch.randn(1, 3, 224, 224, device=DEV)
    with FlopCounterMode(display=False) as fc: m(x1)
    x = torch.randn(BATCH, 3, 224, 224, device=DEV); n = 10 if DEV == "cuda" else 1
    with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == "cuda")):
        m(x)
        if DEV == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n): m(x)
        if DEV == "cuda": torch.cuda.synchronize()
    r = dict(backbone=label, timm=name, params_M=sum(p.numel() for p in m.parameters()) / 1e6,
             GFLOPs=fc.get_total_flops() / 2e9, img_per_s=n * BATCH / (time.time() - t0), dim=m.num_features)
    del m; torch.cuda.empty_cache() if DEV == "cuda" else None
    return r

COST = [cost("vit_base_patch16_224", "ViT-B/16 (nhánh ViT đóng băng)")] + [cost(n, l) for n, l in BACKBONES.items()]
import pandas as pd
print(pd.DataFrame(COST).round(2).to_string(index=False))


thiết bị: cuda Tesla T4
                      backbone                                          timm  params_M  GFLOPs  img_per_s  dim
ViT-B/16 (nhánh ViT đóng băng)                          vit_base_patch16_224     85.80   17.56     475.67  768
                        Swin-B swin_base_patch4_window7_224.ms_in22k_ft_in1k     86.74   15.43     239.05 1024
                        Swin-T          swin_tiny_patch4_window7_224.ms_in1k     27.52    4.49     521.51  768


## 4. Thư viện phân tích

In [4]:
# ==============================================================================
# 4. THƯ VIỆN PHÂN TÍCH (y nguyên code đã dùng cho Spherical trên ViT / APT)
# ==============================================================================
"""Offline analysis of the dumps written by the patched APT repo (apt_patch.py).
Per seed dir:  train_frozen_{t}.pt {f,y}   test_frozen_{j}.pt {f,y}   test_{t}_{j}.pt {k,y}  (k = logits over the t+1 seen tasks).
Labels are already in task order (class c of task j lies in [lo_j, hi_j)), frozen/KAC rows of a test set are in the same order."""
import os, glob, math
import numpy as np
import torch
import torch.nn.functional as F

W_GRID = [0.25, 0.5, 0.75, 1.0]
KAPPA_GRID = [8, 16, 32, 64]
K_GRID = [1, 3, 5]
LAM_GRID = [0.5, 1.0, 2.0]
WIN_GRID = [0.0, 0.5]


def embed(f):
    return F.normalize(F.layer_norm(f.float(), (f.shape[1],)), dim=1)


def sph_kmeans(Z, k, g, iters=20):
    k = min(k, Z.shape[0])
    C = Z[torch.randperm(Z.shape[0], generator=g)[:k]].clone()
    for _ in range(iters):
        a = (Z @ C.T).argmax(1)
        newC = torch.zeros_like(C).index_add_(0, a, Z)
        empty = newC.norm(dim=1) < 1e-8
        if empty.any():
            newC[empty] = Z[torch.randint(0, Z.shape[0], (int(empty.sum()),), generator=g)]
        C = F.normalize(newC, dim=1)
    return C


def seeds_complete(run_dir):
    res = {}
    for d in sorted(glob.glob(os.path.join(run_dir, "seed*"))):
        T = len(glob.glob(os.path.join(d, "train_frozen_*.pt")))
        if T and all(os.path.exists(os.path.join(d, f"test_{t}_{j}.pt")) for t in range(T) for j in range(t + 1)) \
           and all(os.path.exists(os.path.join(d, f"test_frozen_{j}.pt")) for j in range(T)):
            res[int(os.path.basename(d)[4:])] = (d, T)
    return res


def block_lsm(x, bounds):
    out = torch.empty_like(x)
    for lo, hi in bounds:
        out[:, lo:hi] = F.log_softmax(x[:, lo:hi], 1)
    return out


def block_lse(ls, bounds):
    out = torch.empty_like(ls)
    for lo, hi in bounds:
        out[:, lo:hi] = torch.logsumexp(ls[:, lo:hi], 1, keepdim=True)
    return out


def _metrics(A, T):
    at = [A[:t + 1, t].mean() for t in range(T)]
    forg = np.mean([A[j, j] - A[j, T - 1] for j in range(T - 1)]) if T > 1 else 0.0   # = repo's calc_general_forgetting
    return float(np.mean(at)), float(at[-1]), float(forg)


def analyse_seed(d, T, seed):
    ld = lambda n: torch.load(os.path.join(d, n))
    tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
    te = [ld(f"test_frozen_{j}.pt") for j in range(T)]
    # ranh giới task lấy từ độ rộng logit (repo CUB có nhãn 1..200 nên 1 lớp rỗng, task chứa nó có 19 lớp nhưng vẫn chiếm 20 cột)
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    has = torch.zeros(C, dtype=torch.bool); has[ytr.unique()] = True
    print(f"    {int((~has).sum())} lớp không có ảnh train (bỏ qua):", (~has).nonzero().flatten().tolist())
    z0 = Ztr[:512]
    G = 1 - z0 @ z0.T
    inv_med = 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()
    g = torch.Generator().manual_seed(seed)
    cen = {k: torch.zeros(C, k, Ztr.shape[1]) for k in K_GRID}
    for c in range(C):
        if not has[c]: continue
        Zc = Ztr[ytr == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g)
            cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    acc, tid = {}, np.zeros((T, T))
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te[j]["y"].long()), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = embed(te[j]["f"]); lo, hi = bounds_all[j]
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    return rows, dict(seed=seed, task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"))


def analyse(run_dir):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        print(f"  seed {seed}: {T} task", flush=True)
        r, dg = analyse_seed(d, T, seed); rows += r; diag.append(dg)
    return rows, diag


"""Spherical branch on three feature sources, same KAC logits, same grids (needs fusion_online.py code in the same namespace).
  frozen      : ViT without prompt. Centres = train images of each task (once). Test features = frozen features.
  apt_stale   : APT features (CLS after the prompts). Centre of a class = APT features of its train images taken right after
                its own task and never refreshed; test features = APT features of the CURRENT model (what a real continual run has).
  apt_refresh : at every step, centres rebuilt from the current model's features of ALL seen train images.
                A real continual run does not keep old train images -> reference only (separates drift from feature quality).
Dumps: train_frozen_{t}.pt, train_apt_{t}_{j}.pt, test_frozen_{j}.pt, test_apt_{t}_{j}.pt, test_{t}_{j}.pt (see apt_patch.py)."""

SOURCES = ["frozen", "apt_stale", "apt_refresh"]


def _build_cen(Z, y, C, g):
    has = torch.zeros(C, dtype=torch.bool); has[y.unique()] = True
    cen = {k: torch.zeros(C, k, Z.shape[1]) for k in K_GRID}
    for c in has.nonzero().flatten().tolist():
        Zc = Z[y == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g); cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    return cen, has


def _inv_med(Z):
    z0 = Z[:512]; G = 1 - z0 @ z0.T
    return 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()


def sources_available(d, T):
    ex = lambda n: os.path.exists(os.path.join(d, n))
    out = ["frozen"]
    if all(ex(f"train_apt_{t}_{t}.pt") for t in range(T)) and all(ex(f"test_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)):
        out.append("apt_stale")
        if all(ex(f"train_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)): out.append("apt_refresh")
    return out


def analyse_seed_src(d, T, seed, source):
    ld = lambda n: torch.load(os.path.join(d, n))
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    te_y = [ld(f"test_frozen_{j}.pt")["y"].long() for j in range(T)]
    g = torch.Generator().manual_seed(seed)
    if source == "frozen":
        tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
        zte_cache = [embed(ld(f"test_frozen_{j}.pt")["f"]) for j in range(T)]
        zte = lambda t, j: zte_cache[j]
    else:
        tr = [ld(f"train_apt_{t}_{t}.pt") for t in range(T)]
        zte = lambda t, j: embed(ld(f"test_apt_{t}_{j}.pt")["f"])
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    if source != "apt_refresh":                                   # centres built once, from each task's own train images
        Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
        cen_s, has_s = _build_cen(Ztr, ytr, C, g); inv_s = _inv_med(Ztr)
        if source == "apt_stale":
            fz = [ld(f"train_frozen_{t}.pt")["y"] for t in range(T)]
            assert all(torch.equal(a["y"], b) for a, b in zip(tr, fz)), "train_apt và train_frozen lệch thứ tự"
    acc, tid = {}, np.zeros((T, T)); drift = np.full((T, T), np.nan)
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        if source == "apt_refresh":                               # re-embed everything seen so far with the current model
            parts = [ld(f"train_apt_{t}_{j}.pt") for j in range(t + 1)]
            Zt = torch.cat([embed(x["f"]) for x in parts]); yt = torch.cat([x["y"] for x in parts]).long()
            cen, has = _build_cen(Zt, yt, V, g); inv_med = _inv_med(embed(parts[0]["f"]))
        else:
            cen, has, inv_med = cen_s, has_s, inv_s
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te_y[j]), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = zte(t, j); lo, hi = bounds_all[j]
            if source == "apt_stale" and j < t:                   # how far did the same test images move since task j?
                drift[j, t] = (z * zte(j, j)).sum(1).mean().item()
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, source=source, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    dg = dict(seed=seed, source=source,
              task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"),
              cos_same_image_old_task_last=float(np.nanmean([drift[j, T - 1] for j in range(T - 1)])) if (source == "apt_stale" and T > 1) else float("nan"))
    return rows, dg


def analyse_all(run_dir, sources=SOURCES):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        have = sources_available(d, T)
        for s in sources:
            if s not in have:
                print(f"  seed {seed}: thiếu dump cho nguồn {s} (cần APT_DUMP_APT={'2' if s == 'apt_refresh' else '1'}) - bỏ qua"); continue
            print(f"  seed {seed}: {T} task, nguồn {s}", flush=True)
            r, dg = analyse_seed_src(d, T, seed, s); rows += r; diag.append(dg)
    return rows, diag


## 5. Chạy

In [5]:
# ==============================================================================
# 5. CHẠY: mỗi (dataset, seed) -> thư mục đặc trưng Swin (cùng định dạng dump của APT) -> phân tích
#    Có dump KAC của phiên APT (Add Input) : Swin proto/fuse/hier + ViT proto/fuse/hier + KAC + oracle, cùng ảnh, cùng nhãn
#    Không có dump                          : chỉ Swin proto (không cần KAC)
# ==============================================================================
import shutil, time
import pandas as pd

def find_dumps(ds, seed):
    for d in sorted(glob.glob(f"{INPUT_ROOT}/**/runs/kac/{ds}/{PHASE}/dumps/seed{seed}", recursive=True)):
        T = len(glob.glob(f"{d}/train_frozen_*.pt"))
        if T and all(os.path.exists(f"{d}/test_{t}_{j}.pt") for t in range(T) for j in range(t + 1)) \
             and all(os.path.exists(f"{d}/test_frozen_{j}.pt") for j in range(T)):
            return d, T
    return None, None

def write_swin_dir(ds, seed, dump, name):
    tr, tr_y, ev, ev_y = seed_split(ds, seed)
    msg = check_against_json(ds, seed, ev, ev_y)
    Fd = features(ds, name)
    b = np.cumsum([0] + DS_SPEC[ds]["sizes"]); T = len(b) - 1
    d = f"{OUT}/work/{ds}/{PHASE}/{BACKBONES[name]}/seed{seed}"; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    for t in range(T):
        for name, items, ys in (("train", tr, tr_y), ("test", ev, ev_y)):
            sel = [n for n, y in enumerate(ys) if b[t] <= y < b[t + 1]]          # cùng thứ tự với repo APT (lọc theo task, giữ thứ tự json)
            f = torch.stack([Fd[items[n][0]][items[n][1]] for n in sel])
            torch.save({"f": f, "y": torch.tensor([ys[n] for n in sel])}, f"{d}/{name}_frozen_{t}.pt")
    if dump:
        for t in range(T):
            for j in range(t + 1): os.symlink(f"{dump}/test_{t}_{j}.pt", f"{d}/test_{t}_{j}.pt")
    return d, T, msg

def analyse_proto_only(d, T, seed, ds):
    """Spherical-Swin thuần khi không có dump KAC (cùng cách tính với analyse_seed_src: cùng tâm, cùng κ)."""
    ld = lambda n: torch.load(os.path.join(d, n))
    b = np.cumsum([0] + DS_SPEC[ds]["sizes"]); bounds = [(int(b[t]), int(b[t + 1])) for t in range(T)]; C = int(b[-1])
    tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
    Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
    g = torch.Generator().manual_seed(seed)
    cen, has = _build_cen(Ztr, ytr, C, g); inv_med = _inv_med(Ztr)
    te = [ld(f"test_frozen_{j}.pt") for j in range(T)]
    acc, tid = {}, np.zeros((T, T))
    for t in range(T):
        V = bounds[t][1]
        for j in range(t + 1):
            z, y = embed(te[j]["f"]), te[j]["y"].long(); lo, hi = bounds[j]
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2); s[:, ~has[:V]] = -1e4
                    acc.setdefault(("fuse", 1.0, kp, k), np.zeros((T, T)))[j, t] = 100.0 * (s.argmax(1) == y).float().mean().item()
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        rows.append(dict(seed=seed, source="frozen", variant="fuse", w=1.0, kappa=key[2], k=key[3], lam=np.nan, w_in=np.nan,
                         avg=avg, last=last, forgetting=forg))
    return rows, dict(seed=seed, source="frozen", task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])),
                      cos_same_image_old_task_last=float("nan"))

ROWS, DIAG = [], []
def add(r, dg, ds, bb, has):
    global ROWS
    for x in r: x.update(dataset=ds, backbone=bb, has_kac=has)
    dg.update(dataset=ds, backbone=bb, has_kac=has); ROWS += r; DIAG.append(dg)

for ds in DATASETS:
    for seed in SEEDS:
        t0 = time.time()
        dump, Td = find_dumps(ds, seed)
        print(f"== {ds} | seed {seed} | dump KAC: {dump or 'KHÔNG CÓ -> chỉ Swin proto'}", flush=True)
        if dump:                                                              # ViT đóng băng: đọc thẳng từ dump
            r, dg = analyse_seed_src(dump, Td, seed, "frozen"); add(r, dg, ds, "ViT", True)
        for name, bb in BACKBONES.items():
            d, T, msg = write_swin_dir(ds, seed, dump, name)
            if dump:
                assert T == Td, f"{ds} seed {seed}: dump có {Td} task, split có {T}"
                r, dg = analyse_seed_src(d, T, seed, "frozen")
            else:
                r, dg = analyse_proto_only(d, T, seed, ds)
            add(r, dg, ds, bb, bool(dump))
        print(f"   split: {msg} | xong sau {time.time() - t0:.0f}s", flush=True)
pd.DataFrame(ROWS).to_csv(f"{OUT}/rows_all_{PHASE}.csv", index=False)


== CIFAR100 | seed 0 | dump KAC: /kaggle/input/datasets/thnhtranhuynh/cifar-seed0-dumps/apt_spherical/runs/kac/CIFAR100/val/dumps/seed0
CIFAR100: /kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python | train 50000 ảnh, test 10000 ảnh
CIFAR100/tr/Swin-B: trích 50000 ảnh ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

swin_base_patch4_window7_224.ms_in22k_ft_in1k: 1024 chiều | tiền xử lý (3, 224, 224), crop_pct 0.9
   128/50000 ảnh, 3s
   6528/50000 ảnh, 32s
   12928/50000 ảnh, 62s
   19328/50000 ảnh, 95s
   25728/50000 ảnh, 125s
   32128/50000 ảnh, 157s
   38528/50000 ảnh, 188s
   44928/50000 ảnh, 219s
CIFAR100/tr/Swin-T: trích 50000 ảnh ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

swin_tiny_patch4_window7_224.ms_in1k: 768 chiều | tiền xử lý (3, 224, 224), crop_pct 0.9
   128/50000 ảnh, 1s
   6528/50000 ảnh, 15s
   12928/50000 ảnh, 30s
   19328/50000 ảnh, 45s
   25728/50000 ảnh, 59s
   32128/50000 ảnh, 73s
   38528/50000 ảnh, 87s
   44928/50000 ảnh, 101s
   split: khớp file json của phiên APT | xong sau 560s
== CIFAR100 | seed 1 | dump KAC: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cifar/apt_spherical/runs/kac/CIFAR100/val/dumps/seed1
   split: khớp file json của phiên APT | xong sau 36s
== CIFAR100 | seed 2 | dump KAC: /kaggle/input/notebooks/meomeomeo124/fork-of-apt-kac-spherical-cifar/apt_spherical/runs/kac/CIFAR100/val/dumps/seed2
   split: khớp file json của phiên APT | xong sau 40s
== CUB200 | seed 0 | dump KAC: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cub/apt_spherical/runs/kac/CUB200/val/dumps/seed0
CUB200: /kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder | train 5994 ảnh, test 5794 ảnh
CUB200/tr/Swin-B: trích 5994 ảnh ..

## 6. Bảng kết quả

In [6]:
# ==============================================================================
# 6. BẢNG KẾT QUẢ: chọn cấu hình theo Last trung bình các seed (trên Val), ViT và Swin cạnh nhau
# ==============================================================================
df = pd.DataFrame(ROWS)
df.loc[(df.variant == "fuse") & (df.w == 1.0), "variant"] = "proto"       # fuse với w = 1 chính là Spherical thuần
df.loc[df.variant.isin(["KAC", "oracle"]), "backbone"] = "-"               # KAC/oracle không phụ thuộc backbone Spherical
df = df.drop_duplicates(subset=["dataset", "seed", "backbone", "variant", "w", "kappa", "k", "lam", "w_in"])
CFG = ["w", "kappa", "k", "lam", "w_in"]
ORDER = [("-", "KAC")] + [(b, v) for b in ["ViT"] + list(BACKBONES.values()) for v in ("proto", "fuse", "hier")] + [("-", "oracle")]

def best_table(sub):
    g = sub.groupby(["backbone", "variant"] + CFG, dropna=False).agg(
        n=("seed", "nunique"), seeds=("seed", lambda s: ",".join(map(str, sorted(s.unique())))),
        avg=("avg", "mean"), avg_sd=("avg", "std"), last=("last", "mean"), last_sd=("last", "std"),
        forg=("forgetting", "mean"), forg_sd=("forgetting", "std")).reset_index()
    out = []
    for bb, v in ORDER:
        x = g[(g.backbone == bb) & (g.variant == v)]
        if len(x): out.append(x.loc[x["last"].idxmax()])
    return pd.DataFrame(out)

fmt = lambda m, s: f"{m:.2f}" + (f" ± {s:.2f}" if pd.notna(s) else "")
TABLES = []
for ds in DATASETS:
    sub = df[df.dataset == ds]
    paired = sub[sub.has_kac]
    if len(paired):
        t = best_table(paired); t.insert(0, "dataset", ds); t.insert(1, "nhóm", "có KAC (so sánh cặp)"); TABLES.append(t)
        print(f"\n### {ds} — các seed có dump KAC: {sorted(int(s) for s in paired.seed.unique())}")
        for _, r in t.iterrows():
            cfg = " ".join(f"{c}={r[c]:g}" for c in CFG if pd.notna(r[c]) and not (r.variant == "proto" and c == "w"))
            print(f"  {r.backbone:7}{r.variant:7} Avg {fmt(r.avg, r.avg_sd):15} Last {fmt(r['last'], r.last_sd):15} Forg {fmt(r.forg, r.forg_sd):14} {cfg}")
    if len(paired) == 0 or set(sub.seed) != set(paired.seed):              # có seed thiếu dump: báo riêng proto trên mọi seed
        for bb in BACKBONES.values():
            t2 = best_table(sub[(sub.backbone == bb) & (sub.variant == "proto")]); t2.insert(0, "dataset", ds)
            t2.insert(1, "nhóm", f"{bb} proto, mọi seed"); TABLES.append(t2); r = t2.iloc[0]
            print(f"  {bb} proto trên mọi seed ({r.seeds}): Avg {fmt(r.avg, r.avg_sd)} | Last {fmt(r['last'], r.last_sd)} | Forg {fmt(r.forg, r.forg_sd)}")

dg = pd.DataFrame(DIAG).groupby(["dataset", "backbone"]).task_id_acc_old_last.agg(["mean", "std", "count"]).round(2)
print("\nĐộ chính xác chọn đúng task của Spherical (k=1, κ_mult=16; task cũ, sau task cuối):\n", dg)
pd.concat(TABLES).to_csv(f"{OUT}/swin_vs_vit_best_{PHASE}.csv", index=False, float_format="%.4f")
dg.to_csv(f"{OUT}/task_id_acc_{PHASE}.csv")
print("\nĐã lưu:", f"{OUT}/swin_vs_vit_best_{PHASE}.csv", f"{OUT}/rows_all_{PHASE}.csv", f"{OUT}/task_id_acc_{PHASE}.csv")

pd.DataFrame(COST).to_csv(f"{OUT}/backbone_cost.csv", index=False, float_format="%.2f")
print("\nChi phí backbone (forward thuần, fp16):\n", pd.DataFrame(COST).round(2).to_string(index=False))



### CIFAR100 — các seed có dump KAC: [0, 1, 2]
  -      KAC     Avg 92.33 ± 0.60    Last 88.01 ± 0.72    Forg 3.65 ± 1.11    
  ViT    proto   Avg 86.11 ± 0.29    Last 80.28 ± 0.50    Forg 6.26 ± 0.39    kappa=16 k=5
  ViT    fuse    Avg 92.95 ± 0.49    Last 89.08 ± 0.60    Forg 3.60 ± 0.76    w=0.25 kappa=32 k=5
  ViT    hier    Avg 90.85 ± 0.42    Last 85.01 ± 0.40    Forg 6.14 ± 0.46    kappa=8 k=5 lam=0.5 w_in=0
  Swin-B proto   Avg 89.31 ± 0.51    Last 84.22 ± 0.53    Forg 5.23 ± 0.46    kappa=16 k=5
  Swin-B fuse    Avg 93.58 ± 0.52    Last 89.83 ± 0.57    Forg 3.47 ± 0.56    w=0.5 kappa=16 k=5
  Swin-B hier    Avg 92.24 ± 0.57    Last 87.29 ± 0.51    Forg 5.04 ± 0.71    kappa=8 k=5 lam=0.5 w_in=0
  Swin-T proto   Avg 78.13 ± 0.18    Last 70.14 ± 0.39    Forg 8.81 ± 0.36    kappa=16 k=5
  Swin-T fuse    Avg 92.76 ± 0.54    Last 88.64 ± 0.61    Forg 3.54 ± 0.93    w=0.25 kappa=16 k=5
  Swin-T hier    Avg 87.91 ± 0.64    Last 79.75 ± 0.52    Forg 8.31 ± 1.23    kappa=8 k=5 lam=0.5

In [7]:
import zipfile
z = f"/kaggle/working/swin_spherical_results_{PHASE}.zip" if OUT.startswith("/kaggle") else f"{OUT}/results_{PHASE}.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
    for p in glob.glob(f"{OUT}/*.csv"): f.write(p, os.path.basename(p))
print("Tải về:", z)


Tải về: /kaggle/working/swin_spherical_results_val.zip
